# Ch 14 — 손실 곡선과 체크포인트

손실 곡선 5가지 패턴을 진단하고, 재개 가능한 체크포인트 인프라를 구축한다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part4/ch14_loss_curves.ipynb)

In [ ]:
# !pip install -q torch tokenizers datasets

import json
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'device: {device}')
print(f'torch:  {torch.__version__}')

## GPTMini 모델 정의

Ch 12/13 과 동일한 모델.

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 8000
    n_layer: int = 6
    n_head: int = 8
    d_model: int = 320
    max_len: int = 512

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head; self.d_model = cfg.d_model
        self.qkv  = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)
        hd = C // self.n_head
        q = q.view(B, T, self.n_head, hd).transpose(1, 2)
        k = k.view(B, T, self.n_head, hd).transpose(1, 2)
        v = v.view(B, T, self.n_head, hd).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).contiguous().view(B, T, C))

class FFN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        h = int(cfg.d_model * 8 / 3)
        self.gate = nn.Linear(cfg.d_model, h, bias=False)
        self.up   = nn.Linear(cfg.d_model, h, bias=False)
        self.down = nn.Linear(h, cfg.d_model, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = nn.RMSNorm(cfg.d_model); self.attn = CausalSelfAttention(cfg)
        self.norm2 = nn.RMSNorm(cfg.d_model); self.ffn  = FFN(cfg)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class GPTMini(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg    = cfg
        self.embed  = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos    = nn.Embedding(cfg.max_len, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm   = nn.RMSNorm(cfg.d_model)
        self.head   = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
    def forward(self, x, y=None):
        B, T = x.shape
        h = self.embed(x) + self.pos(torch.arange(T, device=x.device))
        for block in self.blocks:
            h = block(h)
        logits = self.head(self.norm(h))
        loss = None
        if y is not None:
            loss = F.cross_entropy(logits.view(-1, self.cfg.vocab_size), y.view(-1))
        return logits, loss

cfg = GPTConfig()
print(f'모델 파라미터: {sum(p.numel() for p in GPTMini(cfg).parameters())/1e6:.1f}M')

def get_batch(batch_size=4, seq_len=64, vocab_size=8000):
    x = torch.randint(0, vocab_size, (batch_size, seq_len))
    y = torch.randint(0, vocab_size, (batch_size, seq_len))
    return x.to(device), y.to(device)

## 최소 예제 — 로깅 + 시각화

학습 중 jsonl 로 실시간 기록 → matplotlib 으로 EMA 시각화.

In [ ]:
class Logger:
    def __init__(self, path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.f = self.path.open("a", buffering=1)   # line buffering
        self.start = time.time()

    def log(self, **kw):
        kw["t"] = round(time.time() - self.start, 1)
        self.f.write(json.dumps(kw) + "\n")

    def close(self):
        self.f.close()

print('Logger 정의 완료')

In [ ]:
# 짧은 학습 실행 + 로깅
model  = GPTMini(cfg).to(device)
opt    = torch.optim.AdamW(model.parameters(), lr=6e-4, betas=(0.9, 0.95))
logger = Logger('/tmp/runs/exp_demo/loss.jsonl')
model.train()

for step in range(300):
    x, y = get_batch()
    _, loss = model(x, y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    opt.zero_grad(set_to_none=True)

    if step % 50 == 0:
        logger.log(step=step, loss=loss.item(), lr=opt.param_groups[0]['lr'])
        print(f'  step {step:4d} | loss {loss.item():.3f}')

logger.close()
print('\n로그 저장 완료: /tmp/runs/exp_demo/loss.jsonl')

In [ ]:
# 시각화 — 미니 dashboard
with open('/tmp/runs/exp_demo/loss.jsonl') as f:
    rows = [json.loads(l) for l in f]

steps = [r['step'] for r in rows]
losses = [r['loss'] for r in rows]

# EMA smoothing — 노이즈 제거
def ema(xs, alpha=0.05):
    s, out = xs[0], []
    for x in xs:
        s = alpha * x + (1 - alpha) * s
        out.append(s)
    return out

plt.figure(figsize=(8, 4))
plt.plot(steps, losses, alpha=0.3, label='raw')
plt.plot(steps, ema(losses), label='ema')
plt.xlabel('step'); plt.ylabel('loss')
plt.axhline(math.log(8000), color='gray', linestyle='--', label=f'ln(8000)={math.log(8000):.2f}')
plt.legend(); plt.title('학습 손실 곡선'); plt.tight_layout()
plt.show()

## 손실 곡선 5가지 패턴 시각화

| 패턴 | 진단 |
|---|---|
| **정상** | warmup ↓, cosine 따라 부드럽게 ↓ |
| **발산** | NaN 또는 폭발 ↑ |
| **정체** | ln(vocab) 부근에 머무름 |
| **스파이크** | 부드럽게 가다 한 번에 ↑ |
| **과적합** | train ↓ but val ↑ |

In [ ]:
import numpy as np

steps_n = np.arange(200)

# 5가지 패턴을 시뮬레이션으로 시각화
def cosine_decay(steps, start=9.0, end=2.5, warmup=20):
    vals = []
    for s in steps:
        if s < warmup:
            vals.append(start - (start - (start * 0.9)) * s / warmup)
        else:
            progress = (s - warmup) / (len(steps) - warmup)
            vals.append(end + (start - end) * 0.5 * (1 + math.cos(math.pi * progress)))
    return np.array(vals) + np.random.randn(len(steps)) * 0.05

fig, axes = plt.subplots(1, 5, figsize=(16, 3))

# 1. 정상
axes[0].plot(steps_n, cosine_decay(steps_n)); axes[0].set_title('정상')
axes[0].set_xlabel('step'); axes[0].set_ylabel('loss')

# 2. 발산
diverge = np.ones(200) * 9.0
diverge[50:] = 9.0 + np.arange(150) * 0.3
diverge[100:] = float('nan')
axes[1].plot(steps_n, diverge, color='red'); axes[1].set_title('발산 (NaN)')
axes[1].set_xlabel('step')

# 3. 정체
plateau = np.ones(200) * math.log(8000) + np.random.randn(200) * 0.02
axes[2].plot(steps_n, plateau, color='orange'); axes[2].set_title('정체')
axes[2].axhline(math.log(8000), color='gray', linestyle='--', alpha=0.5)
axes[2].set_xlabel('step')

# 4. 스파이크
spike = cosine_decay(steps_n)
spike[80] = spike[80] + 4.0
spike[130] = spike[130] + 2.5
axes[3].plot(steps_n, spike, color='purple'); axes[3].set_title('스파이크')
axes[3].set_xlabel('step')

# 5. 과적합
train_loss = cosine_decay(steps_n, start=9.0, end=1.5)
val_loss   = cosine_decay(steps_n, start=9.0, end=3.5)
val_loss[100:] += np.linspace(0, 1.5, 100)  # val 반등
axes[4].plot(steps_n, train_loss, label='train')
axes[4].plot(steps_n, val_loss, label='val', color='orange')
axes[4].set_title('과적합'); axes[4].legend()
axes[4].set_xlabel('step')

plt.suptitle('손실 곡선 5가지 패턴', fontsize=13)
plt.tight_layout()
plt.show()

## 실전 — 재개 가능한 체크포인트

저장할 5가지: model + optimizer + scheduler + step + RNG state.

In [ ]:
def save_ckpt(path, model, optimizer, scheduler, step, scaler=None):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    state = {
        'step':      step,                         # step 도 같이 — scheduler 가 같은 자리에서 재개
        'model':     model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'rng_torch': torch.get_rng_state(),
        'rng_cuda':  torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }
    if scaler is not None:
        state['scaler'] = scaler.state_dict()
    torch.save(state, path)
    print(f'  saved ckpt at step {step}: {path}')

def load_ckpt(path, model, optimizer, scheduler, scaler=None):
    state = torch.load(path, map_location=device)
    model.load_state_dict(state['model'])
    optimizer.load_state_dict(state['optimizer'])
    scheduler.load_state_dict(state['scheduler'])  # lr 곡선이 끊긴 자리부터
    torch.set_rng_state(state['rng_torch'])
    if state['rng_cuda'] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state['rng_cuda'])
    if scaler and 'scaler' in state:
        scaler.load_state_dict(state['scaler'])
    return state['step']

print('save_ckpt / load_ckpt 정의 완료')

In [ ]:
# 자동 저장 + 재개 패턴
ckpt_dir  = Path('/tmp/runs/exp_resume')
last_ckpt = ckpt_dir / 'last.pt'

model2    = GPTMini(cfg).to(device)
opt2      = torch.optim.AdamW(model2.parameters(), lr=6e-4, betas=(0.9, 0.95))
total_steps_r = 400
warmup_r  = 50

def lr_lambda_r(s):
    if s < warmup_r: return s / warmup_r
    p = (s - warmup_r) / (total_steps_r - warmup_r)
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * p))

sched2  = torch.optim.lr_scheduler.LambdaLR(opt2, lr_lambda_r)
logger2 = Logger(ckpt_dir / 'loss.jsonl')

start_step = 0
if last_ckpt.exists():                                                  # 시작 시 last.pt 있으면 자동 재개
    start_step = load_ckpt(last_ckpt, model2, opt2, sched2)
    print(f'  resumed from step {start_step}')
else:
    print(f'  fresh start')

model2.train()
for step in range(start_step, total_steps_r):
    x, y = get_batch()
    _, loss = model2(x, y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model2.parameters(), 1.0)
    opt2.step()
    sched2.step()
    opt2.zero_grad(set_to_none=True)

    if step % 50 == 0:
        logger2.log(step=step, loss=loss.item(), lr=opt2.param_groups[0]['lr'])

    if step > 0 and step % 100 == 0:                                    # 100 step 마다 단계별 체크포인트
        save_ckpt(ckpt_dir / f'step_{step:06d}.pt', model2, opt2, sched2, step)
        save_ckpt(last_ckpt, model2, opt2, sched2, step)                # last.pt 는 매번 덮어쓰기

logger2.close()
print(f'\n학습 완료. last.pt: {last_ckpt}')

In [ ]:
# 재개 테스트 — last.pt 에서 로드 후 step 확인
model_resume = GPTMini(cfg).to(device)
opt_resume   = torch.optim.AdamW(model_resume.parameters(), lr=6e-4)
sched_resume = torch.optim.lr_scheduler.LambdaLR(opt_resume, lr_lambda_r)

if last_ckpt.exists():
    resumed_step = load_ckpt(last_ckpt, model_resume, opt_resume, sched_resume)
    print(f'재개 step: {resumed_step}')
    print(f'재개 시점 lr: {opt_resume.param_groups[0]["lr"]:.6f}')
    print(f'기대 lr (step={resumed_step}): {lr_lambda_r(resumed_step) * 6e-4:.6f}')
else:
    print('last.pt 없음 — 위 셀을 먼저 실행하세요')

## 연습문제

1. 본인 학습을 100 step 돌려 jsonl 로깅을 켜고 §최소 예제의 plot 으로 시각화. raw vs EMA 곡선 비교.
2. 일부러 lr 을 `1e-2` 로 키워 발산을 발생시켜라. 손실 곡선과 NaN 시점을 기록.
3. 학습 도중 (Ctrl+C 로) 중단하고 `last.pt` 에서 재개. step 과 lr 이 정확히 이어지는지 확인.
4. step / RNG / optimizer 중 **하나만** 저장 안 하고 재개해보라. 어떤 증상?
5. **(생각해볼 것)** "손실 곡선이 부드럽게 떨어지면 학습 잘 됐다" 가 항상 옳은가? **부드러우면서 모델은 망가지는** 시나리오는?

In [ ]:
# 연습 1: jsonl 로깅 + EMA 시각화


In [ ]:
# 연습 2: lr=1e-2 으로 발산 유도


In [ ]:
# 연습 3: 중단 후 재개 실험


In [ ]:
# 연습 4: 부분 저장 후 재개 증상 관찰


In [ ]:
# 연습 5: 자유 답변
